In [4]:
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.feature import OneHotEncoder, StringIndexer
from pyspark.ml.regression import DecisionTreeRegressor
from pyspark.ml.feature import RFormula
from pyspark.ml.evaluation import RegressionEvaluator
import pandas as pd
import click
import os



In [5]:
spark = SparkSession.builder \
    .appName("MLApp") \
    .getOrCreate()

25/07/20 21:25:04 WARN Utils: Your hostname, RealElvo resolves to a loopback address: 127.0.1.1; using 192.168.100.10 instead (on interface wlp2s0)
25/07/20 21:25:04 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/07/20 21:25:06 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [8]:
from dotenv import load_dotenv
load_dotenv()

project_root = os.getenv("PROJECT_ROOT")

file_path = os.path.join(project_root, "data", "ML_Data.parquet")

airbnb_df = spark.read.parquet(file_path)

airbnb_df.select("neighbourhood_cleansed", "room_type","bedrooms","bathrooms","number_of_reviews","price").show(10)


+----------------------+---------------+--------+---------+-----------------+-----+
|neighbourhood_cleansed|      room_type|bedrooms|bathrooms|number_of_reviews|price|
+----------------------+---------------+--------+---------+-----------------+-----+
|      Western Addition|Entire home/apt|     1.0|      1.0|            180.0|170.0|
|        Bernal Heights|Entire home/apt|     2.0|      1.0|            111.0|235.0|
|        Haight Ashbury|   Private room|     1.0|      4.0|             17.0| 65.0|
|        Haight Ashbury|   Private room|     1.0|      4.0|              8.0| 65.0|
|      Western Addition|Entire home/apt|     2.0|      1.5|             27.0|785.0|
|      Western Addition|Entire home/apt|     2.0|      1.0|             31.0|255.0|
|               Mission|   Private room|     1.0|      1.0|            647.0|139.0|
|          Potrero Hill|   Private room|     1.0|      1.0|            453.0|135.0|
|               Mission|Entire home/apt|     2.0|      1.0|            320.0

In [9]:
# split the data into training and test sets
train_df, test_df = airbnb_df.randomSplit([.8,.2],seed=42)

print(f"Training data count: {train_df.count()}")
print(f"Test data count: {test_df.count()}")

25/07/20 21:33:52 WARN package: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Training data count: 5780


Test data count: 1366


In [10]:
# transform the train dataframe using VectorAssembler
vecassembler = VectorAssembler(
    inputCols=["bedrooms"],
    outputCol="features"
)
vec_train_df = vecassembler.transform(train_df)
vec_train_df.select("bedrooms", "features","price").show(10)


+--------+--------+-----+
|bedrooms|features|price|
+--------+--------+-----+
|     1.0|   [1.0]|200.0|
|     1.0|   [1.0]|130.0|
|     1.0|   [1.0]| 95.0|
|     1.0|   [1.0]|250.0|
|     3.0|   [3.0]|250.0|
|     1.0|   [1.0]|115.0|
|     1.0|   [1.0]|105.0|
|     1.0|   [1.0]| 86.0|
|     1.0|   [1.0]|100.0|
|     2.0|   [2.0]|220.0|
+--------+--------+-----+
only showing top 10 rows

